# Part 2: Testing Volterra Theory on MNIST Random Features

**COMP 450 Project** - Kaan Altas & Yigit Karasu

## Objective

Test if the Volterra integral equation predicts SGD dynamics when the feature matrix A comes from **real MNIST images** (with planted targets).

## Approach: Option A (Planted Targets)

- Use real MNIST images to construct A = σ(X @ W)
- Generate synthetic target x* and compute b = A @ x*
- This preserves the paper's generative assumption: b = Ax* + noise

## Key Parameters

| Parameter | Value |
|-----------|-------|
| n (samples) | 5000 |
| r (aspect ratios) | 0.5, 1.0, 1.2 |
| d (features) | 2500, 5000, 6000 |
| Noise (R_tilde) | 0 (noiseless) |

## Frozen Conventions

**These are locked and must not change:**

- **Loss:** f(x) = (1/2n)||Ax - b||²
- **LR mapping:** η_code = γ_paper / n
- **Hessian:** H = (1/n)A^T A
- **γ_max_theory:** (2/r) / mean(λ) — Paper's Theorem 1.2
- **γ_max_safe:** 2 / λ_max — Conservative bound
- **Batch size:** 1 (single-sample SGD)
- **Sampling:** With replacement

In [ ]:
import sys
import os

# Add parent directory to path
sys.path.append(os.path.abspath(os.path.join('..')))  

import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# Import project modules
from src.data_loader import load_mnist
from src.random_features import generate_random_weights, compute_features
from src.spectral import compute_eigenvalues, marchenko_pastur_density
from src.sgd import LeastSquaresSGD, StreamingSGD
from src.volterra import VolterraSolver

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## Step 2: MNIST Data Loading & Preprocessing Diagnostics

In [ ]:
# Global parameters (optimized for ~5-10 min runtime)
n = 5000
num_epochs = 20  # Reduced from 50 (enough to see convergence)
num_runs = 2     # Reduced from 5 (still shows variance)

# Load MNIST
print("Loading MNIST...")
X_mnist, y_mnist = load_mnist(root='../data', train=True, flatten=True, 
                               subset_size=n, download=False)
X_mnist = X_mnist.to(device)
print(f"X_mnist shape: {X_mnist.shape}")
print(f"X_mnist dtype: {X_mnist.dtype}")
print(f"X_mnist device: {X_mnist.device}")

In [ ]:
# MNIST Feature Statistics
print("\n=== MNIST Feature Statistics ===")
print(f"Global mean: {X_mnist.mean().item():.4f}")
print(f"Global std: {X_mnist.std().item():.4f}")
print(f"Min: {X_mnist.min().item():.4f}, Max: {X_mnist.max().item():.4f}")

# Per-column statistics
col_means = X_mnist.mean(dim=0)
col_stds = X_mnist.std(dim=0)
print(f"\nPer-column mean: min={col_means.min().item():.4f}, max={col_means.max().item():.4f}")
print(f"Per-column std: min={col_stds.min().item():.4f}, max={col_stds.max().item():.4f}")

# Check for NaN/Inf
print(f"\nNaN values: {torch.isnan(X_mnist).sum().item()}")
print(f"Inf values: {torch.isinf(X_mnist).sum().item()}")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Per-column means
axes[0].hist(col_means.cpu().numpy(), bins=50, alpha=0.7)
axes[0].set_xlabel('Column Mean')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Per-Column Means')
axes[0].axvline(x=0, color='r', linestyle='--', label='Zero')
axes[0].legend()

# Per-column stds
axes[1].hist(col_stds.cpu().numpy(), bins=50, alpha=0.7)
axes[1].set_xlabel('Column Std')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Per-Column Stds')

# Sample images
sample_idx = np.random.choice(n, 9, replace=False)
for i, idx in enumerate(sample_idx):
    ax = plt.subplot(3, 9, 19 + i)
    img = X_mnist[idx].cpu().numpy().reshape(28, 28)
    ax.imshow(img, cmap='gray')
    ax.axis('off')
    ax.set_title(f'{y_mnist[idx].item()}')

plt.tight_layout()
plt.show()

---
## Step 3: Random Features Construction

Build A = σ(X @ W / √784) using shifted ReLU.

**GPT's Tweak #2:** Apply per-column centering (not just global mean).

In [ ]:
def build_random_features(X, r, device):
    """
    Builds random features A = σ(XW) with per-column centering.
    
    Returns:
        A: Centered feature matrix
        W: Random weight matrix (for streaming SGD)
        diagnostics: Dict with centering stats
    """
    n_samples = X.shape[0]
    d_in = X.shape[1]  # 784
    d_out = int(n_samples * r)
    
    print(f"\n=== Building Random Features (r={r}) ===")
    print(f"Input: X ∈ R^({n_samples} × {d_in})")
    print(f"Output: A ∈ R^({n_samples} × {d_out})")
    
    # Generate random weights
    W = generate_random_weights(d_in, d_out).to(device)
    
    # Compute features
    A = compute_features(X, W, activation='shifted_relu', scale_by_sqrt_d=True)
    
    # GPT's Tweak #2: Per-column centering
    col_means_before = A.mean(dim=0)
    max_col_mean_before = col_means_before.abs().max().item()
    print(f"Before centering: max |column mean| = {max_col_mean_before:.4f}")
    
    A = A - A.mean(dim=0, keepdim=True)  # Per-column centering
    
    col_means_after = A.mean(dim=0)
    max_col_mean_after = col_means_after.abs().max().item()
    print(f"After centering: max |column mean| = {max_col_mean_after:.6f}")
    
    # Feature statistics
    print(f"\nFeature matrix A statistics:")
    print(f"  Mean: {A.mean().item():.6f}")
    print(f"  Std: {A.std().item():.4f}")
    print(f"  Min: {A.min().item():.4f}, Max: {A.max().item():.4f}")
    
    diagnostics = {
        'max_col_mean_before': max_col_mean_before,
        'max_col_mean_after': max_col_mean_after,
        'A_mean': A.mean().item(),
        'A_std': A.std().item()
    }
    
    return A, W, diagnostics

---
## Step 4: Generate Planted Targets

Generate x* with ||x*||² = 1, compute b = A @ x*.

**GPT's Fix:** Compute initial loss explicitly, don't assume 0.5.

In [ ]:
def generate_planted_targets(A, device):
    """
    Generate planted targets b = A @ x* with ||x*|| = 1.
    
    Returns:
        x_star: Ground truth solution
        b: Target vector
        x0: Initial point (zeros)
        initial_loss: f(x0) = (1/2n)||b||²
    """
    n, d = A.shape
    
    # Generate normalized x*
    x_star = torch.randn(d, device=device)
    x_star = x_star / torch.norm(x_star)  # ||x*||² = 1
    
    # Planted targets
    b = A @ x_star
    
    # Initial point
    x0 = torch.zeros(d, device=device)
    
    # GPT's Fix: Compute initial loss explicitly
    initial_loss = (1.0 / (2 * n)) * torch.sum(b**2).item()
    
    print(f"\n=== Planted Targets ===")
    print(f"||x*||² = {torch.norm(x_star).item()**2:.4f} (should be 1.0)")
    print(f"||b||² = {torch.norm(b).item()**2:.4f}")
    print(f"Initial loss f(x0) = {initial_loss:.4f}")
    print(f"NOTE: Initial loss may differ from 0.5 depending on A's scaling")
    
    return x_star, b, x0, initial_loss

---
## Step 5: Eigenvalue Computation & Scaling

**GPT's Tweak #3:** Scale A so mean(λ) ≈ 1 for cleaner LR comparisons.

Compute both γ_max_theory and γ_max_safe.

In [ ]:
def analyze_and_scale_spectrum(A, r, n):
    """
    Compute eigenvalues, scale A, and compute both gamma_max values.
    """
    print(f"\n=== Eigenvalue Analysis ===")
    
    # Initial eigenvalues
    print("Computing eigenvalues (this may take a moment)...", flush=True)
    eigvals_initial = compute_eigenvalues(A)
    mean_eig_initial = np.mean(eigvals_initial)
    
    print(f"Before scaling:")
    print(f"  mean(λ) = {mean_eig_initial:.4f}")
    print(f"  λ_max = {np.max(eigvals_initial):.4f}")
    print(f"  λ_min = {np.min(eigvals_initial):.6f}")
    
    # GPT's Tweak #3: Scale A so mean(λ) ≈ 1
    scale_factor = 1.0 / np.sqrt(mean_eig_initial)
    A_scaled = A * scale_factor
    print(f"\nScaling A by {scale_factor:.4f} to normalize mean(λ) → 1")
    
    # Recompute eigenvalues after scaling
    print("Recomputing eigenvalues after scaling...", flush=True)
    eigvals = compute_eigenvalues(A_scaled)
    mean_eig = np.mean(eigvals)
    max_eig = np.max(eigvals)
    min_eig = np.min(eigvals[eigvals > 1e-10])  # Smallest non-zero
    std_eig = np.std(eigvals)
    
    print(f"\nAfter scaling:")
    print(f"  mean(λ) = {mean_eig:.4f} (should be ≈1)")
    print(f"  λ_max = {max_eig:.4f}")
    print(f"  λ_min (non-zero) = {min_eig:.6f}")
    print(f"  std(λ) = {std_eig:.4f}")
    
    # Compute both gamma_max values
    gamma_max_theory = (2.0 / r) / mean_eig
    gamma_max_safe = 2.0 / max_eig
    
    # Spectral ratio (key diagnostic)
    spectral_ratio = max_eig / mean_eig
    
    print(f"\n=== Critical Step Sizes ===")
    print(f"  γ_max_theory = {gamma_max_theory:.4f} (Theorem 1.2, mean-based)")
    print(f"  γ_max_safe = {gamma_max_safe:.4f} (λ_max-based, conservative)")
    print(f"  λ_max / mean(λ) = {spectral_ratio:.2f}")
    
    if spectral_ratio > 5:
        print(f"  ⚠️ WARNING: Heavy spectral tail detected!")
    elif spectral_ratio > 2:
        print(f"  ⚠️ CAUTION: Moderate spectral spread.")
    else:
        print(f"  ✓ Spectrum is well-behaved (ratio < 2).")
    
    spectral_info = {
        'mean_eig': mean_eig,
        'max_eig': max_eig,
        'min_eig': min_eig,
        'std_eig': std_eig,
        'spectral_ratio': spectral_ratio,
        'scale_factor': scale_factor
    }
    
    return A_scaled, eigvals, gamma_max_theory, gamma_max_safe, spectral_info

In [ ]:
def plot_spectrum_comparison(eigvals, r, title_suffix=""):
    """
    Plot eigenvalue histogram with Marchenko-Pastur overlay.
    """
    fig, ax = plt.subplots(figsize=(10, 5))
    
    # MNIST eigenvalues
    ax.hist(eigvals, bins=50, density=True, alpha=0.7, 
            label=f'MNIST Random Features', color='steelblue')
    
    # Marchenko-Pastur theoretical bounds
    lambda_plus = (1 + np.sqrt(r))**2
    lambda_minus = (1 - np.sqrt(r))**2
    
    # MP density
    x = np.linspace(max(0.001, lambda_minus * 0.5), lambda_plus * 1.5, 200)
    mp_density = marchenko_pastur_density(x, r, sigma_sq=1.0)
    ax.plot(x, mp_density, 'r-', lw=2, label=f'Marchenko-Pastur (r={r})')
    
    # Bounds
    ax.axvline(x=lambda_minus, color='r', linestyle='--', alpha=0.5, label=f'λ_min MP = {lambda_minus:.2f}')
    ax.axvline(x=lambda_plus, color='r', linestyle='--', alpha=0.5, label=f'λ_max MP = {lambda_plus:.2f}')
    ax.axvline(x=np.max(eigvals), color='orange', linestyle=':', lw=2, label=f'λ_max actual = {np.max(eigvals):.2f}')
    
    ax.set_xlabel('Eigenvalue', fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    ax.set_title(f'Eigenvalue Distribution {title_suffix}', fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

---
## Step 6: Run SGD + Volterra Comparison

**GPT's Tweak #1:** Two sweeps — Safe (λ_max-based) and Theory (mean-based).

In [ ]:
def run_experiment_sweep(A, b, x_star, eigvals, gamma_max, r, n, num_epochs, num_runs, 
                         sweep_name="", W=None):
    """
    Run SGD + Volterra comparison for multiple learning rates.
    """
    d = A.shape[1]
    steps = n * num_epochs
    multipliers = [1/4, 1/2, 0.9]  # Reduced from 4 to 3 multipliers
    
    print(f"\n=== Running {sweep_name} Sweep (γ_max = {gamma_max:.4f}) ===")
    print(f"    Steps per run: {steps:,} | Runs: {num_runs}")
    
    results = {}
    
    for i, mult in enumerate(multipliers):
        gamma = gamma_max * mult
        key = f"{mult:.4f}"
        print(f"\n  [{i+1}/{len(multipliers)}] γ = {gamma:.4f} ({mult:.2%} of γ_max)")
        
        # --- Volterra Theory ---
        print(f"      Computing Volterra...", end=" ", flush=True)
        R_val = 1.0
        solver = VolterraSolver(eigvals, gamma, r, R=R_val, R_tilde=0.0)
        psi, t_theory = solver.solve(t_max=num_epochs, dt=0.05)
        print(f"Done!")
        
        # --- Empirical SGD ---
        print(f"      Running SGD ({num_runs} runs)...", flush=True)
        sgd_runs = []
        for run in range(num_runs):
            print(f"        Run {run+1}/{num_runs}...", end=" ", flush=True)
            model = LeastSquaresSGD(A, b, learning_rate=gamma/n, batch_size=1)
            loss_hist = model.train(steps)
            sgd_runs.append(loss_hist)
            print(f"final loss = {loss_hist[-1]:.4f}")
        
        sgd_mean = np.mean(sgd_runs, axis=0)
        sgd_std = np.std(sgd_runs, axis=0)
        
        if sgd_mean[-1] > sgd_mean[0] * 10:
            print(f"      ⚠️ SGD DIVERGED!")
        
        results[key] = {
            'gamma': gamma,
            'mult': mult,
            't_theory': t_theory,
            'psi': psi,
            'sgd_mean': sgd_mean,
            'sgd_std': sgd_std
        }
    
    return results

---
## Step 7: Visualization

In [ ]:
def plot_comparison(results_safe, results_theory, r, num_epochs, n, 
                    gamma_max_safe, gamma_max_theory, spectral_ratio):
    """
    Plot SGD vs Volterra comparison for both sweeps.
    """
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))  # 2x3 grid for 3 multipliers
    
    multipliers = [1/4, 1/2, 0.9]  # Match the experiment
    
    for row, (results, sweep_name, gamma_max) in enumerate([
        (results_safe, 'Safe (λ_max)', gamma_max_safe),
        (results_theory, 'Theory (mean)', gamma_max_theory)
    ]):
        for col, mult in enumerate(multipliers):
            ax = axes[row, col]
            key = f"{mult:.4f}"
            data = results[key]
            
            t_steps = np.linspace(0, num_epochs, len(data['sgd_mean']))
            
            # Volterra
            ax.plot(data['t_theory'], data['psi'], 'g-', lw=2.5, 
                    label='Volterra', alpha=0.9)
            
            # SGD mean with std band
            ax.plot(t_steps, data['sgd_mean'], 'orange', lw=2, label='SGD')
            ax.fill_between(t_steps, 
                           data['sgd_mean'] - data['sgd_std'],
                           data['sgd_mean'] + data['sgd_std'],
                           color='orange', alpha=0.3)
            
            ax.set_yscale('log')
            ax.set_xlabel('Epochs')
            ax.set_ylabel('Loss')
            ax.set_title(f"{sweep_name}: γ = {data['gamma']:.3f}")
            ax.grid(True, alpha=0.3)
            ax.legend(loc='upper right')
    
    plt.suptitle(f'MNIST Random Features (r={r}, n={n}, λ_max/mean(λ)={spectral_ratio:.2f})', 
                 fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

---
## Main Experiment Loop

Run experiments for all three aspect ratios: r ∈ {0.5, 1.0, 1.2}

In [ ]:
# Store all results
all_results = {}
aspect_ratios = [0.5, 1.0, 1.2]

for idx, r in enumerate(aspect_ratios):
    print(f"\n{'='*60}")
    print(f"EXPERIMENT {idx+1}/3: r = {r}")
    print(f"{'='*60}")
    
    # Step 3: Build random features
    print("\n[Step 3] Building random features...")
    A, W, feat_diag = build_random_features(X_mnist, r, device)
    
    # Step 4: Generate planted targets
    print("\n[Step 4] Generating planted targets...")
    x_star, b, x0, initial_loss = generate_planted_targets(A, device)
    
    # Step 5: Eigenvalue analysis and scaling
    print("\n[Step 5] Computing eigenvalues and scaling...")
    A_scaled, eigvals, gamma_max_theory, gamma_max_safe, spectral_info = \
        analyze_and_scale_spectrum(A, r, n)
    
    # Recompute b after scaling A
    b_scaled = A_scaled @ x_star
    initial_loss_scaled = (1.0 / (2 * n)) * torch.sum(b_scaled**2).item()
    print(f"\nAfter scaling: Initial loss f(x0) = {initial_loss_scaled:.4f}")
    
    # Plot spectrum
    print("\n[Step 5b] Plotting spectrum...")
    plot_spectrum_comparison(eigvals, r, title_suffix=f"(r={r})")
    
    # Step 6: Run both sweeps
    print("\n[Step 6] Running experiment sweeps...")
    results_safe = run_experiment_sweep(
        A_scaled, b_scaled, x_star, eigvals, 
        gamma_max_safe, r, n, num_epochs, num_runs,
        sweep_name="Safe", W=W
    )
    
    results_theory = run_experiment_sweep(
        A_scaled, b_scaled, x_star, eigvals,
        gamma_max_theory, r, n, num_epochs, num_runs,
        sweep_name="Theory", W=W
    )
    
    # Step 7: Plot comparison
    print("\n[Step 7] Generating comparison plots...")
    plot_comparison(results_safe, results_theory, r, num_epochs, n,
                   gamma_max_safe, gamma_max_theory, spectral_info['spectral_ratio'])
    
    # Store results
    all_results[r] = {
        'feat_diag': feat_diag,
        'spectral_info': spectral_info,
        'gamma_max_theory': gamma_max_theory,
        'gamma_max_safe': gamma_max_safe,
        'initial_loss': initial_loss_scaled,
        'results_safe': results_safe,
        'results_theory': results_theory
    }
    
    print(f"\n✓ Completed r = {r}")

---
## Summary & Interpretation

In [ ]:
print("\n" + "="*60)
print("SUMMARY")
print("="*60)

for r in [0.5, 1.0, 1.2]:
    res = all_results[r]
    print(f"\nr = {r}:")
    print(f"  d = {int(n * r)}")
    print(f"  Per-column centering: max |mean| before = {res['feat_diag']['max_col_mean_before']:.4f}")
    print(f"  Spectral ratio λ_max/mean(λ) = {res['spectral_info']['spectral_ratio']:.2f}")
    print(f"  γ_max_theory = {res['gamma_max_theory']:.4f}")
    print(f"  γ_max_safe = {res['gamma_max_safe']:.4f}")
    print(f"  Initial loss = {res['initial_loss']:.4f}")

## Conclusions

### Key Findings:

1. **Does Volterra predict SGD on MNIST-derived features?**
   - [To be filled based on results]

2. **Spectral behavior:**
   - [Compare to Marchenko-Pastur]
   - [Note any heavy tails or spikes]

3. **Critical step size:**
   - [Does theory-based γ_max cause divergence?]
   - [Is safe γ_max necessary?]

4. **Effect of aspect ratio r:**
   - [r=0.5 vs r=1.0 vs r=1.2]

### Implications for the paper's theory:

- [To be filled based on observed match/mismatch]